# Reliance Inference and Institutional Architecture under Generative AI

## Replication Notebook

This notebook reproduces all numerical results in Sections 3.7-3.8 and the Mathematical Appendix of the paper *"Reliance Inference and Institutional Architecture under Generative AI: A Law-and-Economics Analysis."*

It can be run in Google Colab or any Python 3.9+ environment with the dependencies in `requirements.txt`.

## 1. Setup

Install dependencies and import the simulation modules.

In [ ]:
# Colab setup: clone repository and add code/ to the Python path
# (uncomment when running in Colab)
# !git clone https://github.com/<USERNAME>/reliance-inference-replication.git
# %cd reliance-inference-replication
# import sys, os
# sys.path.insert(0, os.path.join(os.getcwd(), 'code'))

!pip install -q numpy scipy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from model import (Params, posterior_case_iia, normative_fiction,
                   Lambda, q2, social_optimum_effort, four_pillar_table)
from dynamics import DynamicParams, proposition4_check
from sensitivity import (one_at_a_time, monte_carlo, summarize_monte_carlo,
                         correlation_sensitivity)

p = Params()
d = DynamicParams()
print('Parameters initialized.')
print(f'Default: pi_theta={p.pi_theta}, pi_a={p.pi_a}, q_0={p.q0}')

## 2. Proposition 1: The Bayesian Posterior and the Normative Fiction

Section 2.3 derives the Bayesian posterior in Case (ii-a) of the General Approach. The wedge between the legal determination ($\delta_{GA}=1$) and the Bayesian posterior is the *upward normative fiction*.

In [ ]:
post = posterior_case_iia(0.0, p)
fic = normative_fiction(0.0, p)

print(f'At baseline parameters:')
print(f'  Bayesian posterior Pr(theta=1 | s>=s*, a=0) = {post:.4f}')
print(f'  Legal determination delta_GA              = 1.0')
print(f'  Normative fiction magnitude               = {fic:.4f}')

## 3. Lemma 1: Likelihood Ratio is Decreasing in e

$\Lambda(e)$ measures the diagnostic value of observed similarity. Lemma 1 shows it is strictly decreasing in $D$'s filtering effort $e$.

In [ ]:
e_grid = np.linspace(0, p.e_bar, 50)
L = Lambda(e_grid, p)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(e_grid, L, 'b-', linewidth=1.6)
ax.set_xlabel('Filtering effort $e$')
ax.set_ylabel(r'$\Lambda(e)$')
ax.set_title('Lemma 1: Likelihood ratio is decreasing in effort')
ax.grid(alpha=0.3)
plt.show()

## 4. Proposition 3 / Section 3.8: Welfare Decomposition by Regime

Reproduce the four-pillar calibration table from Section 3.8.

In [ ]:
table = four_pillar_table(p)

rows = []
labels = {
    'status_quo': 'Status quo (static rule)',
    'pillar_I': 'Pillar I only',
    'pillar_I_II': 'Pillars I + II',
    'pillar_I_II_III': 'Pillars I + II + III',
    'all_four_pillars': 'All four pillars',
}
for key, lbl in labels.items():
    d_ = table[key]
    rows.append({
        'Regime': lbl,
        'Infringement': round(d_['infringement'], 1),
        'Prevention': round(d_['prevention'], 1),
        'Adjudication': round(d_['adjudication'], 1),
        'AI benefit': round(d_['B'], 1),
        'Welfare W': round(d_['welfare'], 1),
    })

df = pd.DataFrame(rows)
df

## 5. Proposition 4: Institutional Delay

Verify that the static safe-harbor threshold generates cumulative welfare loss bounded below by the quadratic gap formula. Compare against the BAT mechanism (Pillar II).

In [ ]:
res = proposition4_check(p, d)

print(f'Proposition 4 numerical verification:')
print(f'  Static rule total loss   = {res["total_loss_static"]:.3f}')
print(f'  BAT rule total loss      = {res["total_loss_bat"]:.3f}')
print(f'  Quadratic lower bound    = {res["quadratic_bound"]:.3f}')
print(f'  W''(e^SO) at t=0         = {res["W_double_prime_at_e_so"]:.3f}')
print(f'  Bound holds: {res["bound_holds"]}')

In [ ]:
from dynamics import trajectory_pi_theta

t = np.arange(d.T)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

axes[0].plot(t, trajectory_pi_theta(d), 'k-', linewidth=1.6)
axes[0].set_xlabel('Quarter'); axes[0].set_ylabel(r'$\pi_\theta(t)$')
axes[0].set_title('(a) Training-data inclusion'); axes[0].grid(alpha=0.3)

axes[1].plot(t, res['e_so_path'], 'b-', linewidth=1.6, label=r'$e^{SO}_t$')
axes[1].axhline(p.e_star, color='r', linestyle='--', linewidth=1.4,
                label=r'$e^*$ (static)')
axes[1].set_xlabel('Quarter'); axes[1].set_ylabel('Effort')
axes[1].set_title('(b) Optimal vs static effort'); axes[1].legend(); axes[1].grid(alpha=0.3)

discount = np.exp(-d.r * t)
static_loss = discount * (res['W_so_path'] - res['W_static'])
bat_loss = discount * (res['W_so_path'] - res['W_bat'])
axes[2].plot(t, np.cumsum(static_loss), 'r-', linewidth=1.6, label='Static')
axes[2].plot(t, np.cumsum(bat_loss), 'g-', linewidth=1.6, label='BAT')
axes[2].set_xlabel('Quarter'); axes[2].set_ylabel('Cumulative loss')
axes[2].set_title('(c) Discounted cumulative loss'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 6. Appendix B: Correlated Types

Proposition B.1: the normative fiction is strictly increasing in $\rho_{\theta a}$. Verify numerically.

In [ ]:
df_corr = correlation_sensitivity(base_params=p)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(df_corr['rho_theta_a'], df_corr['posterior'], 'b-', linewidth=1.6)
axes[0].axvline(0, color='gray', linestyle=':')
axes[0].set_xlabel(r'$\rho_{\theta a}$'); axes[0].set_ylabel('Posterior')
axes[0].set_title('(a) Bayesian posterior'); axes[0].grid(alpha=0.3)

axes[1].plot(df_corr['rho_theta_a'], df_corr['normative_fiction'], 'r-', linewidth=1.6)
axes[1].axvline(0, color='gray', linestyle=':')
axes[1].set_xlabel(r'$\rho_{\theta a}$'); axes[1].set_ylabel('Fiction magnitude')
axes[1].set_title('(b) Normative fiction'); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 7. Monte Carlo Robustness

Sample parameters from prior distributions and verify that the welfare gain is robust to parameter uncertainty.

In [ ]:
df_mc = monte_carlo(n_draws=1000, seed=42, base_params=p)
summary = summarize_monte_carlo(df_mc)

print('Monte Carlo summary:')
for k, v in summary.items():
    print(f'  {k:<20s}: {v}')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.hist(df_mc['gain_pct'].dropna(), bins=40, color='steelblue',
        edgecolor='white', alpha=0.85)
ax.axvline(summary['mean_gain_pct'], color='red', linestyle='--',
           linewidth=1.5, label=f"Mean: {summary['mean_gain_pct']:.2f}%")
ax.axvline(summary['pct_05'], color='orange', linestyle=':', linewidth=1.2,
           label=f"5%: {summary['pct_05']:.2f}%")
ax.axvline(summary['pct_95'], color='orange', linestyle=':', linewidth=1.2,
           label=f"95%: {summary['pct_95']:.2f}%")
ax.set_xlabel('Welfare gain over status quo (%)')
ax.set_ylabel('Frequency')
ax.set_title(f'Monte Carlo distribution (n = {summary["n"]})')
ax.legend(loc='upper right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. One-at-a-Time Sensitivity

Vary each key parameter and trace the welfare gain as a function of the parameter value.

In [ ]:
from sensitivity import standard_sensitivity_grid

grids = standard_sensitivity_grid(p)

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
labels = {
    'pi_theta': r'$\pi_\theta$',
    'pi_a': r'$\pi_a$',
    'q0': r'$q_0$',
    'q2_0': r'$q_2(0)$',
    'H': r'$H$',
    'c_e_star': r'$c(e^*)$',
}
for ax, (name, lbl) in zip(axes.ravel(), labels.items()):
    df_ = grids[name]
    ax.plot(df_[name], df_['gain_pct'], 'b-', linewidth=1.6)
    ax.axhline(0, color='gray', linewidth=0.6)
    ax.set_xlabel(lbl); ax.set_ylabel('Welfare gain (%)')
    ax.set_title(f'Sensitivity to {lbl}'); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 9. Summary

All results from the manuscript reproduce numerically:

1. **Proposition 1**: Bayesian posterior strictly less than 1 in Case (ii-a).
2. **Lemma 1**: $\Lambda(e)$ strictly decreasing in $e$.
3. **Proposition 3 / Section 3.8**: Welfare monotonically increases across the four pillars; total improvement ~28% over the static status quo.
4. **Proposition 4**: Static rule generates cumulative loss bounded below by the quadratic formula; BAT mechanism reduces this loss by approximately 99%.
5. **Proposition B.1**: Bayesian posterior decreases monotonically in $\rho_{\theta a}$, confirming that positive correlation amplifies the normative fiction.
6. **Monte Carlo robustness**: 100% of parameter draws yield positive welfare gains; mean gain ~31%, with 5%–95% interval [21%, 44%].

These results constitute the numerical foundation for the manuscript's normative claims.